# Original Spam Mail Prediction Notebook (Baseline)
Preserved from your original Colab notebook for direct comparison with the upgraded system below.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [ ]:
# LOADING THE DATA FROM CSV FILE TO PANDAS DATA FRAME
import os
if not os.path.exists('/content/mail_data.csv'):
    fallback_url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
    pd.read_csv(fallback_url, sep='\t', header=None, names=['Category', 'Message']).to_csv('/content/mail_data.csv', index=False)
raw_mail_data = pd.read_csv('/content/mail_data.csv')
mail_data = raw_mail_data.where((pd.notnull(raw_mail_data)), '')
mail_data.head()


In [ ]:
print(mail_data['Category'].value_counts())
mail_data.loc[mail_data['Category'] == 'spam', 'Category'] = 0
mail_data.loc[mail_data['Category'] == 'ham', 'Category'] = 1
x = mail_data['Message']
y = mail_data['Category'].astype('int')
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=3)
feature_extraction = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
x_train_features = feature_extraction.fit_transform(x_train)
x_test_features = feature_extraction.transform(x_test)
model = LogisticRegression()
model.fit(x_train_features, y_train)
print('Baseline Train Accuracy:', accuracy_score(y_train, model.predict(x_train_features)))
print('Baseline Test Accuracy :', accuracy_score(y_test, model.predict(x_test_features)))


---
# Upgraded Model: High-Accuracy Spam & Phishing Detection System

This upgraded section preserves the original notebook above and adds:
1. **Higher-Accuracy Spam vs. Ham Classifier** — Word + Character N-gram TF-IDF (`sublinear_tf=True`) + handcrafted message/URL features + Soft-Voting Ensemble (`LogisticRegression` + calibrated `LinearSVC` + `ComplementNB`) with Precision, Recall, F1-score, and ROC-AUC.
2. **3-Class Classifier (`Ham` / `Spam` / `Phishing`)** — Combines `mail_data.csv` with a public phishing email dataset plus short SMS-phishing (smishing) samples.
3. **Explainable Phishing Risk Analyzer** — Inspects URLs, suspicious TLDs, IP links, URL shorteners, brand impersonation, urgency, and credential-harvesting cues alongside ML probabilities.


In [ ]:
import os
import re
from urllib.parse import urlparse

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import LinearSVC

URL_SHORTENERS = {
    'bit.ly', 'tinyurl.com', 't.co', 'goo.gl', 'ow.ly', 'is.gd',
    'buff.ly', 'rb.gy', 'cutt.ly', 'shorturl.at', 'tiny.cc', 'rebrand.ly'
}

SUSPICIOUS_TLDS = (
    '.xyz', '.top', '.club', '.work', '.click', '.link', '.tk', '.ml',
    '.ga', '.cf', '.gq', '.ru', '.cn', '.buzz', '.support', '.online', '.site', '.pw'
)

OFFICIAL_BRAND_DOMAINS = {
    'paypal': ('paypal.com',),
    'amazon': ('amazon.com', 'amazon.co.uk', 'amazon.in', 'amzn.to'),
    'apple': ('apple.com', 'icloud.com'),
    'microsoft': ('microsoft.com', 'live.com', 'outlook.com', 'office.com'),
    'google': ('google.com', 'gmail.com', 'accounts.google.com'),
    'netflix': ('netflix.com',),
    'chase': ('chase.com',),
    'citibank': ('citi.com', 'citibank.com'),
    'wellsfargo': ('wellsfargo.com',),
    'irs': ('irs.gov',),
    'dhl': ('dhl.com',),
    'fedex': ('fedex.com',),
    'usps': ('usps.com',),
    'ups': ('ups.com',),
    'instagram': ('instagram.com',),
    'facebook': ('facebook.com', 'fb.com'),
    'whatsapp': ('whatsapp.com',),
}

URGENCY_PATTERNS = [
    r'\burgent\b', r'\bimmediately\b', r'\b24\s*hours?\b', r'\b48\s*hours?\b',
    r'\bsuspended\b', r'\blocked\b', r'\brestricted\b', r'\bdeactivated\b',
    r'\bunauthorized\b', r'\bunusual\s+activity\b', r'\bsecurity\s+alert\b',
    r'\baction\s+required\b', r'\bfinal\s+notice\b', r'\bwill\s+be\s+(?:closed|terminated|locked)\b',
    r'\bcompromised\b', r'\bexpir(?:e|es|ing|ed)\b'
]

CREDENTIAL_PATTERNS = [
    r'\bverify\s+your\b', r'\bconfirm\s+your\b', r'\bupdate\s+your\b',
    r'\bvalidate\s+your\b', r'\bpassword\b', r'\botp\b', r'\bpin\s+code\b',
    r'\bssn\b', r'\bsocial\s+security\b', r'\bcredit\s+card\b',
    r'\bbank\s+(?:account|details|card)\b', r'\bbilling\s+info(?:rmation)?\b',
    r'\bverify\s+(?:identity|account)\b', r'\bclick\s+(?:here|the\s+link|below)\b',
    r'\blog\s*in\b', r'\bsign\s*in\b', r'\brestore\s+access\b'
]

FINANCIAL_PATTERNS = [
    r'\bwire\s+transfer\b', r'\binvoice\b', r'\brefund\b', r'\btax\b',
    r'\bpayment\s+(?:failed|declined|overdue|pending)\b', r'\baccount\s+(?:number|balance)\b',
    r'\bclaim\s+your\b', r'\bprize\b', r'\bwallet\b', r'\bcrypto\b'
]


def extract_urls(text):
    return re.findall(r'(?:https?://|www\.)[^\s<>"\'\)]+', str(text), flags=re.IGNORECASE)


def analyze_phishing_signals(text):
    text_str = str(text)
    lower = text_str.lower()
    urls = extract_urls(text_str)

    reasons = []
    risk_score = 0

    has_ip_url = 0
    has_shortener = 0
    has_suspicious_tld = 0
    has_at_in_url = 0
    has_brand_impersonation = 0
    has_insecure_http = 0

    domains = []
    for u in urls:
        parsed_u = u if u.lower().startswith('http') else 'http://' + u
        try:
            parsed = urlparse(parsed_u)
            host = (parsed.hostname or '').lower()
        except Exception:
            host = ''
        if host:
            domains.append(host)
        if re.match(r'^(?:\d{1,3}\.){3}\d{1,3}$', host):
            has_ip_url = 1
        if host in URL_SHORTENERS or any(host.endswith('.' + s) for s in URL_SHORTENERS):
            has_shortener = 1
        if any(host.endswith(tld) for tld in SUSPICIOUS_TLDS):
            has_suspicious_tld = 1
        if '@' in u:
            has_at_in_url = 1
        if u.lower().startswith('http://'):
            has_insecure_http = 1

    impersonated_brands = []
    if domains:
        for brand, official_list in OFFICIAL_BRAND_DOMAINS.items():
            if re.search(rf'\b{re.escape(brand)}\b', lower):
                for host in domains:
                    if not any(host == off or host.endswith('.' + off) for off in official_list):
                        has_brand_impersonation = 1
                        impersonated_brands.append(f'{brand} -> {host}')
                        break

    urgency_hits = sum(1 for p in URGENCY_PATTERNS if re.search(p, lower))
    credential_hits = sum(1 for p in CREDENTIAL_PATTERNS if re.search(p, lower))
    financial_hits = sum(1 for p in FINANCIAL_PATTERNS if re.search(p, lower))

    if has_ip_url:
        risk_score += 40
        reasons.append('URL uses a raw IP address instead of a domain name')
    if has_brand_impersonation:
        risk_score += 40
        reasons.append(f"Possible brand impersonation in link ({', '.join(impersonated_brands[:2])})")
    if has_suspicious_tld:
        risk_score += 25
        reasons.append('Link uses a high-risk / suspicious TLD')
    if has_at_in_url:
        risk_score += 30
        reasons.append('URL contains "@" symbol (possible redirect obfuscation)')
    if has_shortener:
        risk_score += 20
        reasons.append('Contains shortened URL hiding the real destination')
    if has_insecure_http and (credential_hits > 0 or urgency_hits > 0):
        risk_score += 15
        reasons.append('Unencrypted HTTP link paired with account/security request')
    if credential_hits > 0:
        risk_score += min(30, credential_hits * 15)
        reasons.append(f'Requests credentials, login, or sensitive verification ({credential_hits} cue(s))')
    if urgency_hits > 0:
        risk_score += min(25, urgency_hits * 12)
        reasons.append(f'Uses urgent / account-suspension threat language ({urgency_hits} cue(s))')
    if financial_hits > 0 and (len(urls) > 0 or urgency_hits > 0 or credential_hits > 0):
        risk_score += min(15, financial_hits * 8)
        reasons.append(f'Contains financial/payment lure ({financial_hits} cue(s))')

    risk_score = min(100, risk_score)
    length = max(len(text_str), 1)
    digits = sum(c.isdigit() for c in text_str)
    uppers = sum(c.isupper() for c in text_str)
    exclamations = text_str.count('!')
    currencies = len(re.findall(r'[$£€₹]', text_str))

    numeric_features = [
        len(urls),
        has_ip_url,
        has_shortener,
        has_suspicious_tld,
        has_at_in_url,
        has_brand_impersonation,
        urgency_hits,
        credential_hits,
        financial_hits,
        digits / length,
        uppers / length,
        min(exclamations, 10),
        min(currencies, 10),
        np.log1p(length),
    ]
    return numeric_features, risk_score, reasons


class MessageFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return np.array([analyze_phishing_signals(text)[0] for text in X], dtype=float)


def build_feature_union(max_word_features=25000, max_char_features=25000):
    return FeatureUnion([
        ('word_tfidf', TfidfVectorizer(
            ngram_range=(1, 2),
            sublinear_tf=True,
            min_df=2,
            max_features=max_word_features,
            strip_accents='unicode',
        )),
        ('char_tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            sublinear_tf=True,
            min_df=3,
            max_features=max_char_features,
        )),
        ('signal_features', Pipeline([
            ('extract', MessageFeatureExtractor()),
            ('scale', MinMaxScaler(clip=True)),
        ])),
    ])


def build_ensemble_pipeline(max_word_features=25000, max_char_features=25000):
    clf_lr = LogisticRegression(C=4.0, class_weight='balanced', max_iter=1000, random_state=42)
    clf_svm = CalibratedClassifierCV(
        estimator=LinearSVC(C=1.0, class_weight='balanced', random_state=42),
        cv=3,
    )
    clf_cnb = ComplementNB(alpha=0.2)

    voting_clf = VotingClassifier(
        estimators=[('lr', clf_lr), ('svm', clf_svm), ('cnb', clf_cnb)],
        voting='soft',
        weights=[2, 3, 1],
    )
    return Pipeline([
        ('features', build_feature_union(max_word_features, max_char_features)),
        ('clf', voting_clf),
    ])

print('Helper functions and feature extractors ready!')


In [ ]:
# Part 1: Upgrade Spam vs. Ham accuracy on mail_data.csv
mail_path = '/content/mail_data.csv'
if not os.path.exists(mail_path):
    print('mail_data.csv not found in /content/ (runtime may have reset). Fetching standard SMS Spam dataset...')
    fallback_url = 'https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv'
    sms_df = pd.read_csv(fallback_url, sep='\t', header=None, names=['Category', 'Message'])
    sms_df.to_csv(mail_path, index=False)

sms_raw = pd.read_csv(mail_path).fillna('')
sms_df = sms_raw[['Category', 'Message']].copy()
sms_df['Category'] = sms_df['Category'].astype(str).str.strip().str.lower()
sms_df['Message'] = sms_df['Message'].astype(str).str.strip()
sms_df = sms_df[sms_df['Category'].isin(['ham', 'spam']) & (sms_df['Message'].str.len() > 0)]

before_dedup = len(sms_df)
sms_clean = sms_df.drop_duplicates(subset=['Message']).reset_index(drop=True)
print(f'Loaded {before_dedup} rows ({len(sms_clean)} unique messages after removing {before_dedup - len(sms_clean)} duplicates).')

X_sms = sms_clean['Message']
y_sms = (sms_clean['Category'] == 'spam').astype(int)

X_tr, X_te, y_tr, y_te = train_test_split(
    X_sms, y_sms, test_size=0.2, random_state=42, stratify=y_sms
)

models_binary = {
    'Original Baseline (TF-IDF + LR)': Pipeline([
        ('tfidf', TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)),
        ('clf', LogisticRegression(max_iter=1000)),
    ]),
    'Upgraded LR (Word+Char+Signals)': Pipeline([
        ('features', build_feature_union(15000, 15000)),
        ('clf', LogisticRegression(C=4.0, class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'Upgraded LinearSVC (Calibrated)': Pipeline([
        ('features', build_feature_union(15000, 15000)),
        ('clf', CalibratedClassifierCV(LinearSVC(C=1.0, class_weight='balanced', random_state=42), cv=3)),
    ]),
    'Upgraded Ensemble (LR + SVM + CNB)': build_ensemble_pipeline(15000, 15000),
}

results = []
for name, pipe in models_binary.items():
    pipe.fit(X_tr, y_tr)
    preds = pipe.predict(X_te)
    probs = pipe.predict_proba(X_te)[:, 1]
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_te, preds),
        'Spam Precision': precision_score(y_te, preds),
        'Spam Recall': recall_score(y_te, preds),
        'Spam F1': f1_score(y_te, preds),
        'ROC-AUC': roc_auc_score(y_te, probs),
    })

results_df = pd.DataFrame(results).sort_values('Spam F1', ascending=False)
print('\n=== Spam vs. Ham Comparison (Deduplicated Stratified Test Set) ===')
print(results_df.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_f1 = cross_val_score(build_ensemble_pipeline(15000, 15000), X_sms, y_sms, cv=cv, scoring='f1')
cv_acc = cross_val_score(build_ensemble_pipeline(15000, 15000), X_sms, y_sms, cv=cv, scoring='accuracy')
print(f'\n5-Fold Stratified CV -> Accuracy: {cv_acc.mean():.4f} (+/- {cv_acc.std():.4f}) | Spam F1: {cv_f1.mean():.4f} (+/- {cv_f1.std():.4f})')


In [ ]:
# Part 2: Download public Phishing Email dataset & combine with mail_data.csv
phish_df = None
sources = [
    'https://huggingface.co/datasets/zefang-liu/phishing-email-dataset/resolve/main/Phishing_Email.csv',
    'https://huggingface.co/datasets/sahith002/phishing-email-dataset/resolve/main/Phishing_Email.csv',
]

for url in sources:
    try:
        print(f'Trying to load phishing dataset from: {url}')
        phish_df = pd.read_csv(url)
        print(f'Successfully loaded {len(phish_df)} rows!')
        break
    except Exception as e:
        print(f'Could not load from {url}: {e}')

if phish_df is None:
    print('Trying Hugging Face datasets library fallback...')
    from datasets import load_dataset
    ds = load_dataset('zefang-liu/phishing-email-dataset', split='train')
    phish_df = ds.to_pandas()

text_col = [c for c in phish_df.columns if 'text' in c.lower()][0]
label_col = [c for c in phish_df.columns if 'type' in c.lower() or 'label' in c.lower()][0]

phish_clean = phish_df[[text_col, label_col]].dropna().copy()
phish_clean.columns = ['Message', 'RawLabel']
phish_clean['Message'] = phish_clean['Message'].astype(str).str.strip()
phish_clean['RawLabel'] = phish_clean['RawLabel'].astype(str).str.strip().str.lower()

phish_clean = phish_clean[
    (phish_clean['Message'].str.lower() != 'empty') &
    (phish_clean['Message'].str.len() >= 15)
].drop_duplicates(subset=['Message'])

phish_clean['Category'] = np.where(
    phish_clean['RawLabel'].str.contains('phish') | (phish_clean['RawLabel'] == '1'),
    'phishing',
    'ham'
)

smishing_samples = [
    "URGENT: Your bank account has been suspended due to unusual activity. Verify immediately at http://secure-bank-verify.xyz/login",
    "USPS Alert: Your package could not be delivered due to an incomplete address. Confirm details here: http://usps-redelivery-track.top/confirm",
    "PayPal Security: We detected an unauthorized sign-in. Restore access within 24 hours: http://paypal-account-check.online/auth",
    "Apple ID Locked: Your iCloud account has been restricted. Sign in now to unlock: http://apple-id-support.click/verify",
    "Netflix: Your payment failed and your membership is on hold. Update your billing card now at http://netflix-billing-update.xyz",
    "Chase Bank Alert: Your debit card is temporarily locked. Click http://192.168.45.12/chase-unlock to verify your PIN and restore access.",
    "IRS Final Notice: You have an unclaimed tax refund. Confirm your SSN and bank details at http://irs-refund-claim.site/portal",
    "Amazon Security: An order of $899 was placed on your account. If this wasn't you, cancel immediately: http://bit.ly/amz-verify-99",
    "DHL Express: Customs fee of $2.99 is required to release your parcel. Pay now at http://dhl-parcel-fee.tk/pay",
    "Microsoft Account: Unusual login detected from Russia. Confirm your password immediately: http://ms-account-security.work/signin",
    "Bank Alert: Your online banking access will be deactivated in 24 hours. Validate your OTP and account at http://verify-mybank.pw",
    "FedEx: Delivery attempt failed. Update your shipping address within 48 hours or parcel returns: http://tinyurl.com/fedex-reconfirm",
    "WhatsApp Security: Your account is scheduled for suspension. Verify your 6-digit code at http://whatsapp-verify.club/check",
    "Citibank: Suspicious wire transfer of $1,450 pending. Log in immediately to block it: http://citi-fraud-alert.xyz/login",
    "Google Security Alert: Someone tried to change your password. Review activity now: http://google-security-check.ru/verify",
    "Wells Fargo: Action required to keep your checking account active. Confirm your card details at http://wf-secure-portal.online",
]
smishing_df = pd.DataFrame({'Message': smishing_samples, 'Category': 'phishing'})

sms_ham = sms_clean[sms_clean['Category'] == 'ham'][['Message', 'Category']]
sms_spam = sms_clean[sms_clean['Category'] == 'spam'][['Message', 'Category']]
email_ham = phish_clean[phish_clean['Category'] == 'ham'][['Message', 'Category']].sample(
    n=min(4500, (phish_clean['Category'] == 'ham').sum()), random_state=42
)
email_phish = phish_clean[phish_clean['Category'] == 'phishing'][['Message', 'Category']].sample(
    n=min(4500, (phish_clean['Category'] == 'phishing').sum()), random_state=42
)

combined_df = pd.concat([sms_ham, email_ham, sms_spam, email_phish, smishing_df], ignore_index=True)
combined_df = combined_df.drop_duplicates(subset=['Message']).sample(frac=1, random_state=42).reset_index(drop=True)

print('\n=== Combined 3-Class Dataset Distribution ===')
print(combined_df['Category'].value_counts())
print(f'Total unique messages: {len(combined_df)}')


In [ ]:
# Part 3: Train & Evaluate the 3-Class (Ham / Spam / Phishing) Ensemble Model
X_3 = combined_df['Message']
y_3 = combined_df['Category']

X_train_3, X_test_3, y_train_3, y_test_3 = train_test_split(
    X_3, y_3, test_size=0.2, random_state=42, stratify=y_3
)

spam_phish_model = build_ensemble_pipeline(max_word_features=30000, max_char_features=25000)
print('Training 3-class Ham / Spam / Phishing ensemble...')
spam_phish_model.fit(X_train_3, y_train_3)

y_pred_3 = spam_phish_model.predict(X_test_3)
acc_3 = accuracy_score(y_test_3, y_pred_3)
macro_f1 = f1_score(y_test_3, y_pred_3, average='macro')

print(f'\n3-Class Test Accuracy : {acc_3:.4f} ({acc_3 * 100:.2f}%)')
print(f'3-Class Macro F1-Score: {macro_f1:.4f}\n')
print('=== Classification Report ===')
print(classification_report(y_test_3, y_pred_3, digits=4))

labels_order = ['ham', 'spam', 'phishing']
cm = confusion_matrix(y_test_3, y_pred_3, labels=labels_order)
fig, ax = plt.subplots(figsize=(6, 4.5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(labels_order)))
ax.set_yticks(range(len(labels_order)))
ax.set_xticklabels([l.upper() for l in labels_order])
ax.set_yticklabels([l.upper() for l in labels_order])
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
ax.set_title(f'3-Class Confusion Matrix (Acc: {acc_3 * 100:.2f}%)')
for i in range(len(labels_order)):
    for j in range(len(labels_order)):
        ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black', fontweight='bold')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()


In [ ]:
# Part 4: Unified Spam + Phishing Predictor with Explainable Risk Flags
def analyze_message(message, model=spam_phish_model):
    probs = model.predict_proba([message])[0]
    classes = list(model.classes_)
    prob_map = {c: float(p) for c, p in zip(classes, probs)}

    ml_label = max(prob_map, key=prob_map.get)
    _, risk_score, reasons = analyze_phishing_signals(message)

    final_label = ml_label
    if risk_score >= 55 and (prob_map.get('phishing', 0) >= 0.20 or prob_map.get('spam', 0) >= 0.25):
        final_label = 'phishing'
    elif risk_score >= 70:
        final_label = 'phishing'

    if final_label == 'phishing' or risk_score >= 60:
        risk_level = 'HIGH / CRITICAL' if risk_score >= 60 or prob_map.get('phishing', 0) >= 0.75 else 'HIGH'
    elif final_label == 'spam' or risk_score >= 30:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'

    return {
        'prediction': final_label.upper(),
        'ml_prediction': ml_label.upper(),
        'confidence': prob_map[ml_label],
        'probabilities': prob_map,
        'phishing_risk_score': risk_score,
        'risk_level': risk_level,
        'red_flags': reasons,
    }


test_messages = [
    "I've been searching for the right words to thank you for this breather. I promise I won't take your help for granted.",
    "Hi team, please review the Q3 slides before our 3 PM meeting on https://meet.google.com/abc-defg-hij. Thanks!",
    "CONGRATULATIONS! You have won a FREE £1000 cash prize! Text WIN to 80085 now to claim your reward!",
    "Free entry in 2 a wkly comp to win FA Cup final tkts 21st May. Text FA to 87121 to receive entry question!",
    "URGENT: Your PayPal account has been suspended due to unusual login activity. Verify your identity within 24 hours at http://paypal-security-verify.xyz/login or your account will be locked.",
    "Dear customer, your bank debit card is locked. Click http://192.168.1.55/verify-card to confirm your password and restore access immediately.",
]

print('=== Live Test Predictions ===\n')
for idx, msg in enumerate(test_messages, 1):
    res = analyze_message(msg)
    p = res['probabilities']
    print(f"[{idx}] Message: {msg[:95]}{'...' if len(msg) > 95 else ''}")
    print(f"    -> Prediction : {res['prediction']} (Risk: {res['risk_level']} | Phishing Risk Score: {res['phishing_risk_score']}/100)")
    print(f"    -> Probabilities: Ham={p.get('ham', 0):.1%} | Spam={p.get('spam', 0):.1%} | Phishing={p.get('phishing', 0):.1%}")
    if res['red_flags']:
        print(f"    -> Red Flags  : {'; '.join(res['red_flags'])}")
    print()

model_save_path = '/content/spam_phishing_detector.joblib'
joblib.dump(spam_phish_model, model_save_path)
print(f'Saved upgraded 3-class model to: {model_save_path}')
